# പാഠം 12 - ഏജന്റ് സ്‌ക്രാച്ച്പാഡ് ഉപയോഗിച്ചുള്ള ചാറ്റ് ചരിത്രം കുറയ്ക്കൽ

മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് ഉപയോഗിച്ച് ദീര്ഘസംവാദങ്ങളിൽ കോൺടെകസ്റ്റ് എങ്ങനെ നിയന്ത്രിക്കാമെന്ന് ഈ നോട്ട്ബുക്ക് കാണിക്കുന്നു. സംവാദം വളർന്നതോടെ ടോക്കൺ എണ്ണം കൂടുന്നു — ഒടുവിൽ മോഡലിന്റെ കോൺടെകസ്റ്റ് വിൻഡോയുടെ പരിധി തന്നെ കടന്നുപോകുന്നു. നാം ഇതിന് **കോൺടെകസ്റ്റ് സംഗ്രഹ патേൺ**യും സ്ഥിരതയുള്ള സ്മൃതിക്കുള്ള **ഏജന്റ് സ്‌ക്രാച്ച്പാഡ്**ഉം ഉപയോഗിക്കുന്നു.

## നിങ്ങൾ പഠിക്കുന്നതു:
1. **കോൺടെകസ്റ്റ് മാനേജ്മെന്റിന്റെ പ്രാധാന്യം**: ടോക്കൺ പരിധികളും കോൺടെകസ്റ്റ് വിൻഡോകളും മനസ്സിലാക്കൽ
2. **കോൺടെകസ്റ്റ്-അവേയര്‍ ഏജന്റുകൾ**: അവരുടെ തന്റെ സംവാദ കോൺടെകസ്റ്റ് നിയന്ത്രിക്കുന്ന ഏജന്റുകൾ നിർമ്മിക്കൽ
3. **കോൺടെകസ്റ്റ് സംഗ്രഹ പാറ്റേൺ**: സംവാദ ചരിത്രം ചുരുക്കാൻ ടൂളുകൾ ഉപയോഗിക്കൽ
4. **ഏജന്റ് സ്‌ക്രാച്ച്പാഡ്**: കോൺടെകസ്റ്റ് കുറക്കൽ കഴിഞ്ഞും നിലനിൽക്കുന്ന സ്മൃതി

## ആവശ്യങ്ങൾ:
- ഏജൂർ ഓപ്പൺഎഐ സജ്ജീകരണം, പരിസ്ഥിതി വ്യത്യാസങ്ങൾ ക്രമീകരിച്ചിരിക്കുന്നു
- മുൻപത്തെ പാഠങ്ങളിൽ നിന്നുള്ള അടിസ്ഥാന ഏജന്റ് ആശയങ്ങൾക്കുറിച്ചുള്ള അറിവ്


## ക്രമീകരിക്കൽ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## കോൺടെക്സ്റ്റ് മാനേജ്മെന്റ് എന്തുകൊണ്ട് പ്രധാനമാണ്

ഓരോ LLMക്കും പരിധിയുള്ള ഒരു **കോൺടെക്സ്റ്റ് വിൻഡോ** ഉണ്ട് — ഒരു single അഭ്യർത്ഥനയിൽ വളരെ കൂടാതെ ഇത് പ്രോസസ്സ് ചെയ്യാവുന്ന ടോക്കൺസിന്റെ പരമാവധി സംഖ്യ. ഒരു ബഹുയുദ്ധ സംഭാഷണം പുരോഗമിക്കുമ്പോൾ:

- **ടോക്കൺ എണ്ണവ് രേഖീയമായി വർദ്ധിക്കുന്നു** ഓരോ ഉപയോക്തൃ സന്ദേശത്തിലും അസിസ്റ്റന്‍റിന്റെ പ്രതികരണത്തിലും.
- **പ്രോമ്പ്റ്റ് ടോക്കണുകൾ ചെലവിന് പ്രധാനകാരണം** ആണ് കാരണം മുഴുവൻ ചരിത്രവും ഓരോ തവണയും വീണ്ടും അയയ്ക്കപ്പെടുന്നു.
- അവസാനം സംഭാഷണം **കോൺടെക്സ്റ്റ് വിൻഡോയിലൂടെ മുകളിലാകുന്നു**; മോഡൽ അത് either ടൃങ്കേറ്റ് ചെയ്യുകയോ പിശക് കാണിക്കുകയോ ചെയ്യും.

### കോൺടെക്സ്റ്റ് മാനേജ്മെന്റിനുള്ള തന്ത്രങ്ങൾ

| തന്ത്രം | എങ്ങനെ പ്രവർത്തിക്കുന്നു | ട്രേഡ്-ഓഫ് |
|---|---|---|
| **ടൃങ്കേഷൻ** | ഏറ്റവും പഴയ സന്ദേശങ്ങൾ ഒഴിവാക്കുക | മുൻകാല കോൺടെക്സ്റ്റ് നഷ്ടപ്പെടും |
| **സമ്മോക്ഷണം** | പഴയ സന്ദേശങ്ങൾ സംഗ്രഹമേതാക്കുക | ചില വിശദാംശങ്ങൾ നഷ്ടപ്പെടും, പക്ഷേ മുഖ്യപ്പെട്ട കാര്യങ്ങൾ നിലനിർത്തപ്പെടും |
| **സ്ക്രാച്ച്പാഡ് / ബാഹ്യ മെമ്മറി** | സംവാദത്തിന് പുറത്തു പ്രധാന വസ്തുതകൾ സൂക്ഷിക്കുക | ഉപകരണ കോൾകൾ വേണ്ടിവരും, പക്ഷെ ഏതൊരു കിഴിവും സുഖമേറിയുണ്ട് |

ഈ നോട്ട്ബുക്കിൽ ഞങ്ങൾ **സമ്മോക്ഷണം** ഒരു **സ്ക്രാച്ച്പാഡ് ടൂളുമായി** സംയോജിപ്പിച്ചിരിക്കുകയാണ്, അതിലൂടെ ഏജന്റ് സംവാദ ചരിത്രം കുരുക്കിയാലും തുടർച്ച നിലനിർത്താൻ കഴിയുന്നവനാകും.


## ഒരു സാഹചര്യ-അറിയുന്ന ഏജന്റ് സൃഷ്ടിക്കൽ


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## ദൈർഘ്യമേറിയ സംവാദം അനുകരിക്കൽ

സന്ദർഭം എങ്ങനെ കൂടുന്നതാണെന്ന് കാണാൻ നാം ഒരു ബഹു-തവണ സംവാദം കൂടി നടക്കാം. ഏജന്റിന് പ്രധാന വിശദാംശങ്ങൾ (ആരാധനകൾ, ബഡ്ജറ്റ്, യാത്രാ തീയതികൾ) ടേണുകളുടെ ഇടയിൽ സൂക്ഷിച്ച് തുടർച്ചയുള്ളതായിരിക്കും പ്രകടിപ്പിക്കേണ്ടത്.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ഏജന്റ് മുൻകാല സംഭാഷണങ്ങളിൽ നിന്നുള്ള പ്രസക്തി എങ്ങനെ നിലനിർത്തുന്നു എന്ന് ശ്രദ്ധിക്കൂ — ജപ്പാനും, സൂഷിയും, ക്ഷേത്രങ്ങളും, ഫോട്ടോഗ്രഫിയും, $3000 ബജറ്റും, ഒറ്റയായ യാത്രയും, ഏപ്രിൽ കാലയളവും അവന് അറിയാം. ഒരു ചെറിയ സംഭാഷണത്തിൽ ഇത് നന്നായി പ്രവർത്തിക്കുന്നു, പക്ഷേ സംഭാഷണം വിപുലമാകുമ്പോൾ മുഴുവൻ ചരിത്രം വീണ്ടും അയയ്ക്കുന്നത് ചെലവേറിയതാകും.

സന്ദർഭം കൂട്ടിക്കാൻ സംഭാഷണം കൂടുതൽ 턴ുകൾ കൂടി തുടരാം:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## സൗന്ദർഭം സംക്ഷേപിക്കൽ പാറ്റേൺ

സംഭാഷണം വർദ്ധിക്കുന്നതിനോട അനുബന്ധിച്ച്, സമാഹരിച്ചിട്ടുള്ള സൗന്ദർഭത്തെ സങ്കോചിപ്പിച്ച ആകെ രൂപത്തില്‍ മാറ്റാൻ **സംക്ഷേപണ ഉപകരണം** ഉപയോഗിക്കാം. പ്രയോഗം പ്രധാനം ചെയ്യുന്ന മുൻഗണനകൾ രേഖപ്പെടുത്താൻ ഈ ഉപകരണം വിളിക്കാറുണ്ട്, പഴയ സന്ദേശങ്ങൾ ഒഴിവാക്കിയാലും അനിവാര്യമായ വിവരങ്ങൾ സംരക്ഷിക്കപ്പെടും.

ഈ പാറ്റേൺ കൂടുതൽ ബുദ്ധിമുട്ടുള്ള ചരിത്രം കുറയ്ക്കലിന്റെ അടിസ്ഥാന പാളിയാണ്:
1. പ്രയോഗം സംഭാഷണത്തിൽ നിന്നുള്ള മുഖ്യ വസ്തുതകൾ തിരിച്ചറയുന്നു
2. അവ സംരക്ഷിക്കാൻ സംക്ഷേപണ ഉപകരണം വിളിക്കുന്നു
3. പഴയ സന്ദേശങ്ങൾ സുരക്ഷിതമായി നീക്കം ചെയ്യാവുന്നതാണ്, കാരണം സംക്ഷേപം പ്രധാനപ്പെട്ട കാര്യങ്ങളെ ഉൾപ്പെടുത്തുന്നു

താഴെ, പ്രയോഗം പഠിച്ച കാര്യങ്ങളുടെ സംക്ഷിപ്ത സംഗ്രഹം രേഖപ്പെടുത്താൻ വിളിക്കാൻ കഴിയുന്ന `summarize_preferences` ഉപകരണം നമുക്ക് നിർവചിക്കുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## സാരാംശം

ഈ പാഠത്തില്‍ Microsoft Agent Framework ഉപയോഗിച്ച് ദൈര്‍ഘ്യമേറിയ ഏജന്റ് സംഭാഷണങ്ങളില്‍ പശ്ചാത്തലം എങ്ങനെ കൈകാര്യം ചെയ്യാമെന്ന് നിങ്ങള്‍ പഠിച്ചു:

### പ്രധാന ആശയങ്ങള്‍
- **പശ്ചാത്തല വിന്‍ഡോകള്‍ പരിമിതമാണ്** — സംഭാഷണ ചരിത്രത്തിലെ ഓരോ ടോക്കണും പണം ചെലവാകുകയും പരിധിക്കു കീഴിൽ എണ്ണപ്പെടുകയും ചെയ്യുന്നു.
- **സംഗ്രഹണ ഉപകരണങ്ങള്‍** ഏജന്‍റ് സമാഹരിച്ച പശ്ചാത്തലത്തെ ചുരുക്കപ്പെട്ട സാരാംശങ്ങളിലാക്കി ടോക്കണ്‍ ഉപയോഗം കുറയ്ക്കുകയും അനിവാര്യ വിവരങ്ങള്‍ സംരക്ഷിക്കുകയും ചെയ്യുന്നു.
- **ഏജന്റ് സ്‌ക്രാച്ച്‌പാഡുകള്‍** ഏതൊരു സംഭാഷണ കുറവും തരണം ചെയ്തും സ്ഥിരതയുള്ള പുറം സ്മരണ ഒരുക്കുന്നു.

### നിങ്ങള്‍ നിര്‍മിച്ചത്
- ബഹു-തവണകളുള്ള സംഭാഷണങ്ങളില്‍ തുടര്‍ച്ച നിലനിർത്തുന്ന **പശ്ചാത്തല-അറിവുള്ള ഏജന്‍റ്**
- പ്രധാന ഉപയോക്തൃ വിശദാംശങ്ങള്‍ ചുരുക്കം രൂപത്തില്‍ രേഖപ്പെടുത്തുന്ന **സംഗ്രഹണ ഉപകരണം** (`summarize_preferences`)
- പശ്ചാത്തലം നിലനിർത്തലും മാറ്റം കൈകാര്യം ചെയ്യലും കാണിക്കുന്ന **ബഹു-തവണ സംഭാഷണം**

### യാഥാര്‍ത്ഥ്യപ്രയോജനങ്ങള്‍
- **ഉപഭോക്തൃ സേവന ബോട്ടുകള്‍**: ദൈര്‍ഘ്യമേറിയ പിന്തുണാ സെഷനുകളിലെ പാരാമീറ്ററുകള്‍ ഓര്‍ക്കുക
- **സ്വന്തം അസിസ്റ്റന്റുകള്‍**: പശ്ചാത്തലം വീണ്ടും വിശദീകരിക്കാതെ പുരോഗമിക്കുന്ന പദ്ധതികള്‍ ട്രാക്ക് ചെയ്യുക
- **വിദ്യാഭ്യാസ ട്യൂട്ടറുകള്‍**: നിരവധി ഇടപെടലുകളുടെ വഴി വിദ്യാര്‍ത്ഥിയുടെ പുരോഗതി നിലനിർത്തുക

### അടുത്ത ഘട്ടങ്ങള്‍
- ഫയല്‍ അടിസ്ഥാനമുള്ള സ്ഥിരതയോടെ പൂര്‍ണ്ണ സ്‌ക്രാച്ച്‌പാഡ് ടൂള്‍ നടപ്പിലാക്കുക
- സാരാംശീകരണത്തിനുശേഷം സ്വയമേവ ചരിത്ര കുറവ് കൂട്ടുക
- സെമാന്റിക് മെമ്മറി തിരച്ചിലിനായി വെക്ടര്‍ ഡാറ്റാബേസുകള്‍ ചേര്‍ക്കുക
- പൂര്‍ണ പശ്ചാത്തലത്തോടെ ദിവസങ്ങള്‍ക്ക് ശേഷമുള്ള സംഭാഷണങ്ങള്‍ വീണ്ടും ആരംഭിക്കാന്‍ കഴിവുള്ള ഏജന്‍റുകള്‍ നിര്‍മിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസൗകര്യം അറിയിപ്പ്**:  
ഈ പ്രമാണം AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുമ്പോഴും, യന്ത്രപരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ അസാധുതകൾ ഉണ്ടാകാവുന്നതാണ്. മൗലിക ഭാഷയിലുള്ള അസൽ പ്രമാണം അധികാരപരമായ ഉത്ഭവമെന്ന നിലയിൽ പരിഗണിക്കണം. നിർണായകമായ വിവരങ്ങളുടെ വേണ്ടി പ്രൊഫഷണൽ മാനവ പരിഭാഷ പരിഗണിക്കുന്നത് ഉചിതമാണ്. ഈ വിവർത്തനത്തിന്റെ ഉപയോഗത്താൽ ഉണ്ടായ ഏതൊരു തെറ്റായ വ്യാഖ്യാനത്തിനും ഞങ്ങൾ ഉത്തരവാദിയല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
